In [1]:
import os

os.environ.setdefault("SPARK_VERSION", "3.5")

import pandas as pd

from workflow_sifta._0_splits import SPLIT_SEEDS
from workflow_sifta._4_scoring import outcomes
from workflow_prismadv.results.metrics import summary

In [2]:
PROGRAMS = ["sifta", "prismadv", "gepa_column", "gepa_column_fp", "sifta_random", "sifta_fb_stats", "sifta_fc1",
            "sifta_fc3", "sifta_batches6", "sifta_batches12", "sifta_silent3", "sifta_silent5"]
SCENARIOS = {"test_1_cross_new_data": "New Data Batches", "test_2_new_script_train_data": "New Tasks"}

s = summary(outcomes(["imdb"], SPLIT_SEEDS, PROGRAMS))

In [3]:
def table(rows, digits=1):
    return pd.DataFrame({scenario: {name: f"{100 * s.loc[(test, p), 'f1_mean']:.{digits}f} ± "
                                          f"{100 * s.loc[(test, p), 'f1_sd']:.1f}" for p, name in rows.items()}
                         for test, scenario in SCENARIOS.items()})

In [4]:
# Table 4: SIFTA ablation on imdb, macro F1, mean ± SD over the three splits
table({"sifta": "SIFTA full", "sifta_random": "w/o CFPrec selection", "sifta_fb_stats": "w/o graph backtracing",
       "gepa_column": "GEPA[P], opt. for F1", "gepa_column_fp": "GEPA[P], opt. for CFPrec"}, digits=2)

,New Data Batches,New Tasks
SIFTA full,74.16 ± 5.5,81.23 ± 3.0
w/o CFPrec selection,68.00 ± 3.9,74.60 ± 12.8
w/o graph backtracing,61.99 ± 12.7,70.04 ± 7.0
"GEPA[P], opt. for F1",67.42 ± 2.3,68.08 ± 7.8
"GEPA[P], opt. for CFPrec",63.10 ± 5.0,66.78 ± 8.6


In [5]:
# Table C: amount of feedback
table({"sifta_fb_stats": "SIFTA (w/o graph backtracing)", "sifta_fc1": "SIFTA (n_feedback = 1)",
       "sifta_fc3": "SIFTA (n_feedback = 3)", "sifta": "SIFTA (n_feedback = 5)"})

,New Data Batches,New Tasks
SIFTA (w/o graph backtracing),62.0 ± 12.7,70.0 ± 7.0
SIFTA (n_feedback = 1),66.9 ± 5.3,69.6 ± 12.2
SIFTA (n_feedback = 3),71.0 ± 6.7,76.0 ± 4.8
SIFTA (n_feedback = 5),74.2 ± 5.5,81.2 ± 3.0


In [6]:
# Table D: number of observed data batches
table({"prismadv": "Manual", "sifta_batches6": "SIFTA (n_batches = 6)", "sifta_batches12": "SIFTA (n_batches = 12)",
       "sifta": "SIFTA (all 35 batches)"})

,New Data Batches,New Tasks
Manual,65.5 ± 5.9,70.3 ± 12.8
SIFTA (n_batches = 6),60.6 ± 3.7,68.6 ± 11.6
SIFTA (n_batches = 12),70.4 ± 9.8,76.5 ± 8.5
SIFTA (all 35 batches),74.2 ± 5.5,81.2 ± 3.0


In [7]:
# Table K: silent failures mislabeled as safe during optimization
table({"sifta": "SIFTA (0% noise)", "sifta_silent3": "SIFTA (3% noise)", "sifta_silent5": "SIFTA (5% noise)"})

,New Data Batches,New Tasks
SIFTA (0% noise),74.2 ± 5.5,81.2 ± 3.0
SIFTA (3% noise),74.3 ± 6.1,78.9 ± 3.1
SIFTA (5% noise),73.0 ± 5.7,78.2 ± 2.0
